# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [15]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [16]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:3b'



# My Ollama models
DEEPSEEK="deepseek-r1:1.5b"      
LLAMA="llama3.2:3b"  
QWEN="qwen2.5-coder:14b" 

# My frontier models
GPT_NANO="openai/gpt-5-nano"
GPT_5="openai/gpt-5"
GPT_4="openai/gpt-4"
GPT_4_TURBO="openai/gpt-4-turbo"
GEMINI_2_5_FLASH="google/gemini-2.5-flash"
GEMINI_2_5_PRO="google/gemini-2.5-pro"
GEMINI_1_5_PRO="google/gemini-1.5-pro"
GEMINI_1_5_FLASH="google/gemini-1.5-flash"
GEMINI_1_5_PRO_002="google/gemini-1.5-pro-002"
GEMINI_1_5_FLASH_002="google/gemini-1.5-flash-002"
CLAUDE_4_5_SONNET="anthropic/claude-3.5-sonnet"
CLAUDE_4_6_SONNET="anthropic/claude-3.5-sonnet"

In [17]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OR_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if not api_key:
    print("No OpenAI API key found - please check your .env file")
else:
    print("API key found!")

# Frontier model client (OpenAI)
open_router = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

# Local model client (Ollama via OpenAI-compatible endpoint — Day 2 technique)
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

# System prompt: tells the model its role and tone
system_prompt = """You are a helpful technical tutor who explains programming concepts
and code to software engineers learning about LLMs and AI engineering.
When asked a technical question, provide a clear, thorough explanation with examples
where appropriate. Make sure you add visual explainers. Respond in markdown."""

API key found!
Groq API Key exists and begins gsk_


In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [7]:
# Get gpt-4o-mini to answer, with streaming
# Streaming (Day 5 technique): results stream back token by token with a typewriter effect

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

stream = open_router.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

print(f"Answer from {MODEL_GPT} (streaming):\n")
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)

Answer from gpt-4o-mini (streaming):



Certainly! Let's break down the code snippet you provided step by step to understand its functionality and purpose.

### Code Breakdown

```python
yield from {book.get("author") for book in books if book.get("author")}
```

#### 1. **Context of `yield from`:**
- The keyword `yield` is commonly used in Python to create a generator function. When a generator function is called, it returns a generator object without starting execution immediately. Instead, it will execute the function each time the `next()` method is called on the generator object, and it pauses after hitting a `yield` statement.
- The `yield from` statement is a way to yield all values from an iterable (such as a list, set, or any other collection) in one go without needing to loop through it manually. This makes the code cleaner and easier to read.

#### 2. **Set Comprehension:**
- The expression inside the curly braces `{}` in the code is a set comprehension. A set comprehension is similar to a list comprehension, but it creates a set which eliminates duplicate entries.
- In this case, the code iterates over a collection called `books`. For each `book`, it tries to get the value associated with the key `"author"` using `book.get("author")`.

#### 3. **Conditional Filtering:**
- The `if book.get("author")` part acts as a filter. It checks if the `author` value exists and is truthy. If it is not present (returning `None`) or is an empty string, that specific book's author will be ignored.
  
### Purpose of the Code
- The purpose of this line of code is to yield a unique set of authors from a list of books, where only books with an actual `author` (i.e., non-empty and present) contribute to the set.
- By using `yield from`, it efficiently yields each unique author found in the `books` list.

### Example

Let's consider an example to understand how this works:

```python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": "Author 2"},
    {"title": "Book C", "author": None},  # No author
    {"title": "Book D", "author": "Author 1"},  # Duplicate author
    {"title": "Book E", "author": ""}  # Empty author
]

def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Use the generator
for author in unique_authors(books):
    print(author)
```

### Output
```
Author 1
Author 2
```

### Explanation of the Example
- In the example above, we have a list of `books` with various authors.
- The function `unique_authors` utilizes the provided line of code to create a generator that yields unique authors:
  - It skips `"Author C"` since it is `None` and skips the entry that has an empty string.
  - It includes `"Author 1"` and `"Author 2"` but only yields them once due to the nature of sets.

### Visual Explanation

Here's a visual representation of how the flow works:

```
books
 ┌───────────────────────────────────────────┐
 │                                           │
 │   {"title": "Book A", "author": "Author 1"}      │
 │   {"title": "Book B", "author": "Author 2"}      │
 │   {"title": "Book C", "author": None}              │
 │   {"title": "Book D", "author": "Author 1"}      │
 │   {"title": "Book E", "author": ""}                 │
 │                                           │
 └───────────────────────────────────────────┘

 ↓ (Set comprehension with filtering)

set of authors: {"Author 1", "Author 2"}

 ↓ (yield from)

Iterate and yield:
 "Author 1"
 "Author 2"
```

In conclusion, this line of code effectively extracts and yields unique authors from a collection of book dictionaries, while ignoring any entries without a valid author, leveraging both set comprehension and the generator pattern in Python.

In [20]:
# Get Llama 3.2 to answer
# Using Ollama (Day 2 technique): local open-source model via OpenAI-compatible endpoint
# No API charges, data stays on your machine — but less powerful than frontier models

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

stream = groq.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    stream=True
)

print(f"Answer from gpt_oss_120b (GPT — groq):\n")
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)

Answer from gpt_oss_120b (GPT — groq):



## What the line does – step‑by‑step

```python
yield from {book.get("author") for book in books if book.get("author")}
```

| Piece | Meaning |
|------|----------|
| `books` | An *iterable* (usually a list) of dictionaries that represent books. |
| `book.get("author")` | Look up the value under the key `"author"` in a single book dict. `dict.get` returns `None` if the key is missing (instead of raising `KeyError`). |
| `if book.get("author")` | Keep only those books that actually have a non‑falsy author (i.e. not `None`, `""`, `0`, `False`). |
| `{ … for … }` | **Set comprehension** – builds a `set` of all the author values that passed the filter. Sets automatically **deduplicate** values and have *no* guaranteed order. |
| `yield from <iterable>` | A *generator* construct that yields each element of the given iterable *in turn* as if the generator had written a separate `yield` for every element. It also forwards any values sent to the generator (via `.send()`) and propagates `return` values and exceptions. |

Putting it together:

1. **Collect** every non‑empty author from `books` → a *set* of unique author names.  
2. **Iterate** over that set, yielding each author one at a time to the caller of the surrounding generator.

---

## Visual walk‑through

```
books
 ├─ {'title': '1984',      'author': 'George Orwell'}
 ├─ {'title': 'Animal Farm','author': 'George Orwell'}
 ├─ {'title': 'Fahrenheit 451','author': 'Ray Bradbury'}
 ├─ {'title': 'The Road'}               # no author key
 └─ {'title': 'Dune',      'author': None}
```

### 1️⃣ Set comprehension

```
{book.get("author") for book in books if book.get("author")}
```

| iteration | `book.get("author")` | `if` condition | added to set? |
|-----------|----------------------|----------------|---------------|
| 1st       | 'George Orwell'      | truthy         | ✅ |
| 2nd       | 'George Orwell'      | truthy         | ❌ (already in set) |
| 3rd       | 'Ray Bradbury'       | truthy         | ✅ |
| 4th       | None (missing)       | falsy          | ❌ |
| 5th       | None                 | falsy          | ❌ |

**Resulting set** (order is arbitrary):

```python
{'George Orwell', 'Ray Bradbury'}
```

### 2️⃣ `yield from` expands the set

```
yield from {'George Orwell', 'Ray Bradbury'}
```

is equivalent to:

```python
for _author in {'George Orwell', 'Ray Bradbury'}:
    yield _author
```

So the surrounding generator will produce two values, one after the other.

---

## Why write it this way?

| Goal | How the code satisfies it |
|------|---------------------------|
| **Yield each distinct author** | The set removes duplicates automatically. |
| **Skip missing/empty authors** | The `if book.get("author")` guard filters out `None`, `''`, etc. |
| **Keep the function a generator** | `yield from` delegates the iteration to the set, making the outer function a generator without writing an explicit loop. |
| **Forward `send`, `throw`, `close`** | `yield from` also forwards any values sent to the outer generator, which a manual `for …: yield …` loop would not do automatically. |

---

## Full example in context

```python
def unique_authors(books):
    """Generator that yields each distinct, non‑empty author in *books*."""
    # The line we are discussing:
    yield from {book.get("author") for book in books if book.get("author")}

# -------------------------------------------------
# Demo data
books = [
    {"title": "1984",          "author": "George Orwell"},
    {"title": "Animal Farm",   "author": "George Orwell"},
    {"title": "Fahrenheit 451","author": "Ray Bradbury"},
    {"title": "The Road"},                      # missing author
    {"title": "Dune",          "author": None},
]

# Consume the generator
for a in unique_authors(books):
    print(a)
```

**Possible output** (order may vary because a set is unordered):

```
George Orwell
Ray Bradbury
```

---

## What would happen if we changed parts of the line?

| Change | Effect |
|--------|--------|
| Replace `{…}` with `[…]` (list comprehension) | Duplicates are **not** removed; you would get `'George Orwell'` twice. |
| Remove the `if` clause | `None` values (or empty strings) would be added to the set and later yielded. |
| Use `yield` inside a manual loop instead of `yield from` | Functionally the same for simple iteration, but you lose automatic forwarding of `.send()`, `.throw()`, etc. |
| Use `sorted({...})` before `yield from` | Guarantees a deterministic order (alphabetical) at the cost of an extra sorting step. Example: `yield from sorted({ … })`. |

---

## Quick cheat‑sheet

```python
# 1️⃣ Set comprehension (unique, filtered)
unique_authors = {b.get('author') for b in books if b.get('author')}

# 2️⃣ Yield each element – the “delegating” form
def gen():
    yield from unique_authors      # same as: for a in unique_authors: yield a
```

**Key take‑aways**

- **`{…}`** → *set* → unique, unordered collection.  
- **`if book.get("author")`** → guard against missing/falsy values.  
- **`yield from <iterable>`** → clean way to delegate iteration to any iterable (including sets) while preserving generator semantics.

Feel free to ask if you’d like to see a version that preserves order, handles case‑insensitive duplicates, or integrates with `async` generators!

In [30]:
import os
import sys
from openai import OpenAI
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from rich.live import Live

# Load environment variables
load_dotenv()

class SmartLLMClient:
    def __init__(self):
        self.groq_key = os.getenv("GROQ_API_KEY")
        self.ollama_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
        self.groq_url = os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1")
        
        # Routing Keywords - Updated to include "gpt" and "openai" for Groq
        self.cloud_keywords = os.getenv(
            "CLOUD_MODEL_KEYWORDS", 
            "70b,72b,mixtral,groq,gpt,openai"
        ).split(",")
        self.local_keywords = os.getenv(
            "LOCAL_MODEL_KEYWORDS", 
            "8b,7b,qwen,coder,phi,deepseek"
        ).split(",")
        
        # Rich Console for nice output
        self.console = Console()

    def _decide_backend(self, model: str, force_privacy: bool = False):
        """Determines whether to use 'ollama' or 'groq' based on model name."""
        if force_privacy:
            return "ollama"
        
        model_lower = model.lower()
        
        # Check for cloud preferences FIRST
        for keyword in self.cloud_keywords:
            if keyword in model_lower:
                return "groq"
        
        # Check for local preferences
        for keyword in self.local_keywords:
            if keyword in model_lower:
                return "ollama"
        
        # Default fallback to local
        return "ollama"

    def get_client(self, model: str, force_privacy: bool = False):
        """Returns a configured OpenAI client instance."""
        backend = self._decide_backend(model, force_privacy)
        
        if backend == "groq":
            if not self.groq_key:
                raise ValueError("Groq API key not found. Set GROQ_API_KEY in .env")
            return OpenAI(
                base_url=self.groq_url,
                api_key=self.groq_key
            ), "groq"
        else:
            # Ollama doesn't require an API key for local use
            return OpenAI(
                base_url=self.ollama_url,
                api_key="ollama"  # Required by client but ignored by Ollama
            ), "ollama"

    def list_models(self):
        """Prints available models from both Ollama and Groq."""
        self.console.print("\n[bold]📦 Available Models[/bold]\n")
        
        # 1. Check Ollama (Local)
        try:
            local_client = OpenAI(base_url=self.ollama_url, api_key="ollama")
            models = local_client.models.list()
            self.console.print("[green]🏠 Ollama (Local):[/green]")
            for m in models.data:
                self.console.print(f"  • {m.id}")
        except Exception as e:
            self.console.print(f"[red]  Ollama not reachable: {e}[/red]")
            
        # 2. Check Groq (Cloud)
        if self.groq_key:
            try:
                groq_client = OpenAI(base_url=self.groq_url, api_key=self.groq_key)
                models = groq_client.models.list()
                self.console.print(f"\n[blue]☁️ Groq (Cloud):[/blue]")
                for m in models.data:
                    self.console.print(f"  • {m.id}")
            except Exception as e:
                self.console.print(f"[red]  Groq error: {e}[/red]")
        else:
            self.console.print("\n[yellow]⚠️ Groq API Key not set[/yellow]")

    def stream_chat(self, model: str, messages: list, force_privacy: bool = False):
        """Streams response with live Markdown rendering."""
        client, backend = self.get_client(model, force_privacy)
        
        # Print routing info before streaming starts
        self.console.print(f"\n[bold cyan]🔄 Model:[/bold cyan] {model}")
        self.console.print(f"[bold cyan]🔄 Backend:[/bold cyan] {backend.upper()}")
        self.console.print(f"[bold cyan]🔄 Base URL:[/bold cyan] {client.base_url}")
        self.console.print("[dim]Press Ctrl+C to stop generation[/dim]\n")

        full_response = ""
        
        try:
            stream = client.chat.completions.create(
                model=model,
                messages=messages,
                stream=True,
                temperature=0.7
            )
            
            # Use Rich Live to update Markdown in real-time
            with Live("", console=self.console, refresh_per_second=15) as live:
                for chunk in stream:
                    if chunk.choices and chunk.choices[0].delta.content:
                        content = chunk.choices[0].delta.content
                        full_response += content
                        
                        # Render Markdown live
                        markdown = Markdown(full_response)
                        live.update(markdown)
                        
        except KeyboardInterrupt:
            self.console.print("\n[bold yellow]⚠ Generation stopped by user.[/bold yellow]")
        except Exception as e:
            self.console.print(f"\n[bold red]❌ Error:[/bold red] {e}")
        finally:
            # Ensure final newline
            if full_response:
                self.console.print() 

        return full_response


# --- Example Usage ---
if __name__ == "__main__":
    llm = SmartLLMClient()
    
    # 1. List available models (optional)
    # llm.list_models()
    
    # 2. Test with Groq model (works with or without openai/ prefix)
    messages = [{"role": "user", "content": "Say hello in Markdown!"}]
    
    # This NOW works - routes to Groq
    llm.stream_chat(model="openai/gpt-oss-120b", messages=messages)
    
    # This also works - routes to Groq
    # llm.stream_chat(model="gpt-oss-120b", messages=messages)
    
    # This routes to Ollama (local)
    # llm.stream_chat(model="llama3.1:8b", messages=messages)
    
    # Force local even for large models
    # llm.stream_chat(model="llama-3.3-70b-versatile", messages=messages, force_privacy=True)

🔄 Model: openai/gpt-oss-120b

🔄 Backend: GROQ

🔄 Base URL: https://api.groq.com/openai/v1/

Press Ctrl+C to stop generation

Output()

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]
llm.stream_chat(model="openai/gpt-oss-120b", messages=messages)

In [38]:
messages=[
        {
            "role": "user",
            "content": "Explain why fast inference is critical for reasoning models"
        }
    ]
llm.stream_chat(model="llama-3.3-70b-versatile", messages=messages)


🔄 Model: llama-3.3-70b-versatile

🔄 Backend: GROQ

🔄 Base URL: https://api.groq.com/openai/v1/

Press Ctrl+C to stop generation

Output()

'Fast inference is critical for reasoning models for several reasons:\n\n1. **Real-time Decision Making**: In many applications, such as autonomous vehicles, robotics, and healthcare, decisions need to be made in real-time. Fast inference enables reasoning models to process information quickly and make timely decisions, which is crucial for safety, efficiency, and effectiveness.\n2. **Scalability**: As the amount of data and the complexity of models increase, inference time can become a bottleneck. Fast inference allows reasoning models to handle large volumes of data and scale to meet the demands of complex applications.\n3. **User Experience**: In interactive applications, such as chatbots, virtual assistants, and gaming, slow inference can lead to frustrating user experiences. Fast inference ensures that responses are generated quickly, creating a seamless and engaging interaction.\n4. **Energy Efficiency**: Fast inference can help reduce energy consumption, which is essential for b